In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
#extracting the text from the pdf files
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [ ]:
import os

os.chdir(r"D:\Medical-chatbot")
print(os.getcwd())

In [ ]:
extracted_data=load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
import sys
print(sys.executable)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs :List[Document]= []
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(page_content=doc.page_content, 
                     metadata={"source": src})
        )
    return minimal_docs

In [ ]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
#split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    docs = text_splitter.split_documents(minimal_docs)
    return docs

In [ ]:
docs = text_split(minimal_docs)
print(f"Number of chunks: {len(docs)}")

In [ ]:
docs

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
import torch

def download_embeddings():

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
       model_name=model_name,
       model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"})       

    return embeddings

embeddings = download_embeddings()

In [ ]:
embeddings

In [ ]:
vector=embeddings.embed_query("Hello world")
print(vector)

In [ ]:
print("vector length:", len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=r"D:\Medical-chatbot\.env", override=True)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("GOOGLE_API_KEY =", os.getenv("GOOGLE_API_KEY"))

In [ ]:
import os

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY

pc =Pinecone(api_key=pinecone_api_key, environment="us-east1-gcp")


In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east1"
        )
    )

    index = pc.index(index_name)



In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch =  PineconeVectorStore.from_documents(docs,
                                                 embeddings,
                                                   index_name=index_name)


In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
#add more data to the  exsitign pinecine index


dswith=Document(page_content="This is a new document to add to the existing Pinecone index.",
                 metadata={"source": "new_doc.pdf"})

In [ ]:
docsearch.add_documents([dswith])

In [ ]:
retriever= docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs=retriever.invoke("What is acne ?")
retrieved_docs

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt=(
    "you are a medical assistant. You will be provided with some context and a question. Use the context to answer the question. If you don't know the answer, just say you don't know. Do not try to make up an answer." 
    "use the context to answer the question. If you don't know the answer, just say you don't know. Do not try to make up an answer." 
    "use three sentences to answer the question. "
    "keep the answer concise and to the point. "
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
from langchain.chains import create_retrieval_chain
import inspect

print(inspect.signature(create_retrieval_chain))

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = "GOOGLE_API_KEY"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY
)

print("chatModel ready ✅")

In [ ]:
question_answer_chain=create_stuff_documents_chain(
    
    chatModel,
    prompt
)



In [ ]:
retriever = docsearch.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [ ]:
import os

print(os.getenv("GOOGLE_API_KEY")[:20])

In [ ]:
import os

GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print(GOOGLE_API_KEY[:8])

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY
)

print("Gemini connected ✅")

In [ ]:
chatgoogle=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key="YOUR_GOOGLE_API_KEY"
)

response=chatgoogle.invoke("What is anemia? What are the precautions?")
print(response)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print(key[:6], "...", key[-4:])  # should start with AIzaSy

In [ ]:
response = rag_chain.invoke(
    {"input": "What is this document about?"}
)

print(response["answer"])